In [ ]:
# import ml packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# load dataset
# df_train = pd.read_csv('train.csv')
df_train = pd.read_csv("/kaggle/input/forest-cover-type-prediction/train.csv")

# Introduction

Aim in one sentence:
> Predict an integer classification for the forest cover type (the predominant kind of tree cover) from strictly cartographic variables.

In the following, I will first explore the variables, then I will compare different classification models. Hyperparameter tuning will be applied to the best model. Finally, the model will be applied to the test set.

# EDA


In [ ]:
df_train.dtypes

In [ ]:
len(df_train.columns)

In [ ]:
df_train.describe()

In [ ]:
# find missing values in dataset
df_train.isnull().sum()

Summary of the data types and missing values:
- There are 15120.000000 rows and 56.000000 columns in the dataset.
- All variables are numeric.
- There are no missing values in the dataset.

## Check correlation between variables

In [ ]:
# plot correlation matrix
corr = df_train.corr()
plt.figure(figsize=(20,20))
sns.heatmap(corr, annot=True, cmap='coolwarm')


In [ ]:
# for better visualization, we can plot the first 15 features
corr = df_train.iloc[:, 0:15].corr()
plt.figure(figsize=(20,20))
sns.heatmap(corr, annot=True, cmap='coolwarm')


# Data preprocessing

In [ ]:
# split dataset into features and labels
X = df_train.drop('Cover_Type', axis=1)
y = df_train['Cover_Type']

# split dataset into training and testing set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Model training and evaluation

There are some models that we can use to solve this multi-class classification problem:
- Logistic Regression (LogisticRegression)
- K-Nearest Neighbors (KNeighborsClassifier)
- Naive Bayes (GaussianNB, BernoulliNB, MultinomialNB)
- Decision Trees (DecisionTreeClassifier)
- Random Forest (RandomForestClassifier)
- Gradient Boosting (GradientBoostingClassifier)
- Support Vector Machines (SVC, LinearSVC)
- Neural Networks (MLPClassifier)


In [ ]:
from sklearn.metrics import classification_report,accuracy_score,confusion_matrix,precision_score,recall_score,f1_score, roc_auc_score

# write a function to train and test model
def train_test_model(model, model_name, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    model_eval = pd.DataFrame({'Accuracy': [accuracy], 
                                'Precision': [precision], 
                                'Recall': [recall], 
                                'F1 Score': [f1]}, index=[model_name])
    return model_eval
    
    # print('Accuracy: ', accuracy_score(y_test, y_pred))
    # print('Confusion Matrix: ', confusion_matrix(y_test, y_pred))
    # print('Classification Report: ', classification_report(y_test, y_pred))

## Compare multiple models

In [ ]:
# load models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC

# create a dictionary of models
models = {
    'Logistic Regression': LogisticRegression(),
    'KNN': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM': SVC()
}

# train and test models
models_eval = pd.DataFrame()
for model_name, model in models.items():
    models_eval = models_eval.append(train_test_model(model, model_name, X_train, X_test, y_train, y_test))

In [ ]:
models_eval

# Hyperparameter tuning

From the previous section, we can see that the best model is Random Forest. Now, we will try to improve the model by tuning the hyperparameters.

## RandomizedSearchCV

In [ ]:
# hyperparameter tuning for random forest
from sklearn.model_selection import RandomizedSearchCV

# number of trees in random forest
n_estimators = [int(x) for x in np.linspace(start=200, stop=2000, num=10)]

# number of features to consider at every split
max_features = ['auto', 'sqrt']

# maximum number of levels in tree
max_depth = [int(x) for x in np.linspace(10, 100, num=10)]
max_depth.append(None)

# minimum number of samples required to split a node
min_samples_split = [2, 5, 10]

# minimum number of samples required at each leaf node
min_samples_leaf = [1, 2, 4]

# method of selecting samples for training each tree
bootstrap = [True, False]

# create the random grid
random_grid = {'n_estimators': n_estimators,
                'max_features': max_features,
                'max_depth': max_depth,
                'min_samples_split': min_samples_split,
                'min_samples_leaf': min_samples_leaf,
                'bootstrap': bootstrap}

print(random_grid)



In [ ]:
# use the random grid to search for best hyperparameters
# first create the base model to tune
rf = RandomForestClassifier()

# random search of parameters, using 3 fold cross validation,
# search across 100 different combinations, and use all available cores
rf_random = RandomizedSearchCV(estimator=rf, 
                                param_distributions=random_grid, 
                                n_iter=100, 
                                cv=3, 
                                verbose=2, 
                                random_state=42, 
                                n_jobs=-1)

# fit the random search model
rf_random.fit(X_train, y_train)

rf_random.best_params_

In [ ]:
# train random forest with best hyperparameters
rf_best = RandomForestClassifier(n_estimators=800,
                                min_samples_split=2, 
                                min_samples_leaf=1, 
                                max_features='sqrt', 
                                max_depth=70, 
                                bootstrap=False)
# evaluate model
train_test_model(rf_best, 'Random Forest', X_train, X_test, y_train, y_test)

## GridSearchCV

In [ ]:
# use grid search to find the best hyperparameters of random forest based on the results of random search
from sklearn.model_selection import GridSearchCV

# create the parameter grid based on the results of random search
param_grid = {
    'bootstrap': [False],
    'max_depth': [60, 70, 80],
    'max_features': ['sqrt'],
    'min_samples_leaf': [1, 2, 3],
    'min_samples_split': [2, 3, 4],
    'n_estimators': [700, 800, 900]
}

# create a base model
rf = RandomForestClassifier()

# instantiate the grid search model
grid_search = GridSearchCV(estimator=rf,
                            param_grid=param_grid,
                            cv=3,
                            n_jobs=-1,
                            verbose=2)

# fit the grid search to the data
grid_search.fit(X_train, y_train)
grid_search.best_params_

In [ ]:
# train random forest with best hyperparameters
rf_best = RandomForestClassifier(n_estimators=900,
                                min_samples_split=2, 
                                min_samples_leaf=1, 
                                max_features='sqrt', 
                                max_depth=70, 
                                bootstrap=False)
# evaluate model
train_test_model(rf_best, 'Random Forest', X_train, X_test, y_train, y_test)

# Model prediction

In [ ]:
# final_test = pd.read_csv('test.csv')
df_test = pd.read_csv('/kaggle/input/forest-cover-type-prediction/test.csv')

# check the missing values 
np.sum(df_test.isnull().sum())

In [ ]:
# fit the model using df_train 
rf_best.fit(df_train.drop('Cover_Type', axis=1), df_train['Cover_Type'])
# predict the test data
test_pred_y = rf_best.predict(df_test)

In [ ]:
# save predict data to submission file, add ID column to the submission file
submission = pd.DataFrame({'Id': df_test['Id'], 'Cover_Type': test_pred_y})

submission.to_csv('/kaggle/working/submission.csv', index=False)

# Conclusion

The best model of the evaluated ones is Random Forest with the parameters that I found using RandomizedSearchCV and GridSearchCV. 

What can be improved: try other new models?
